# Data aggregation for 2026 matches

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

Load data

In [ ]:
predictions = pd.read_csv("predictions/predictions_2026_group_matches.csv")
predictions.head()

knockout_matches = pd.read_csv("matches/2026_knockout_matches.csv")

Analyse data

In [3]:
predictions = predictions.sort_values(
    by=["group", "confidence"],
    ascending=[True, False]
)
wins = predictions[["group", "predicted_winner"]].copy()

wins["wins"] = 1

group_wins = (
    wins
    .groupby(["group", "predicted_winner"])
    .sum()
    .reset_index()
    .rename(columns={"predicted_winner": "team"})
)

group_wins["rank"] = (
    group_wins
    .groupby("group")["wins"]
    .rank(method="first", ascending=False)
).astype(int)

group_wins = group_wins.sort_values(
    by=["group", "rank"],
    ascending=[True, True]
)

group_wins = group_wins[group_wins["rank"] <= 3]

Output group win predictions as CSV

In [ ]:
print(group_wins)
group_wins.to_csv(
    "predictions/predictions_2026_group_wins.csv",
    index=False
)

   group           team  wins  rank
1      A   South Africa     2     1
2      A    South Korea     2     2
0      A         Mexico     1     3
7      B          TBA_B     3     1
4      B         Canada     1     2
5      B          Qatar     1     3
8      C         Brazil     3     1
9      C          Haiti     2     2
10     C      Scottland     1     3
11     D      Australia     3     1
12     D          TBA_D     2     2
13     D  United States     1     3
15     E        Germany     3     1
16     E    Ivory Coast     2     2
14     E        Ecuador     1     3
18     F    Netherlands     3     1
19     F          TBA_F     2     2
17     F          Japan     1     3
20     G        Belgium     3     1
22     G           Iran     2     2
21     G          Egypt     1     3
24     H          Spain     3     1
25     H        Uruguay     2     2
23     H     Cape Verde     1     3
27     I        Senegal     3     1
26     I         France     2     2
28     I          TBA_I     

Fill knockout matches with group winners

In [5]:
winners = (
    group_wins[group_wins["rank"] == 1]
    .set_index("group")["team"]
    .to_dict()
)

runners_up = (
    group_wins[group_wins["rank"] == 2]
    .set_index("group")["team"]
    .to_dict()
)

third_place = group_wins[group_wins["rank"] == 3][["group", "team"]]

team_confidence = pd.concat([
    predictions[["group", "team1", "confidence"]]
        .rename(columns={"team1": "team"}),

    predictions[["group", "team2", "confidence"]]
        .rename(columns={"team2": "team"})
])

third_place_strength = (
    third_place
    .merge(team_confidence, on=["group", "team"])
    .groupby(["group", "team"])["confidence"]
    .mean()
    .reset_index(name="avg_confidence")
    .sort_values(by="avg_confidence", ascending=False)
)

third_place_strength.head(8)

,group,team,avg_confidence
5,F,Japan,0.816167
7,H,Cape Verde,0.756859
2,C,Scottland,0.744444
0,A,Mexico,0.734861
10,K,TBA_K,0.729500
6,G,Egypt,0.716944
8,I,TBA_I,0.697163
4,E,Ecuador,0.688333


In [6]:
def resolve_team(value):
    if value.startswith("WINNER_"):
        group = value.split("_")[1]
        return winners.get(group, value)

    if value.startswith("RUNNER_UP_"):
        group = value.split("_")[2]
        return runners_up.get(group, value)

    if value.startswith("THIRD_"):
        idx = int(value.split("_")[1]) - 1
        return third_place_strength.loc[idx, "team"]

    return value


knockout_matches.loc[:, "team1"] = knockout_matches["team1"].apply(resolve_team)
knockout_matches.loc[:, "team2"] = knockout_matches["team2"].apply(resolve_team)

Output team predictions of knockout matches

In [ ]:
knockout_matches.to_csv(
    "predictions/predictions_2026_knockout_matches.csv",
    index=False
)

knockout_matches.head(16)

,match_id,team1,team2,date,round
0,M-2026-73,South Korea,Canada,29.6.2026,16 Final
1,M-2026-74,Germany,Mexico,30.6.2026,16 Final
2,M-2026-75,Netherlands,Haiti,30.6.2026,16 Final
3,M-2026-76,Brazil,TBA_F,30.6.2026,16 Final
4,M-2026-77,Senegal,Qatar,1.7.2026,16 Final
5,M-2026-78,Ivory Coast,France,1.7.2026,16 Final
6,M-2026-79,South Africa,Scottland,1.7.2026,16 Final
7,M-2026-80,Croatia,United States,1.7.2026,16 Final
8,M-2026-81,Australia,Ecuador,2.7.2026,16 Final
9,M-2026-82,Belgium,Japan,2.7.2026,16 Final
